In [1]:
from core import utils, database, paths

In [2]:
import pandas as pd

In [ ]:
def analyse_financial_summaries(prompt_text):
    # Győződj meg róla, hogy az 'alap.AI_tools.OLLAMA_MODEL_NAME' létezik és a megfelelő modellnevet tartalmazza
    # Ha nincs ilyen, használd a "gemma" stringet közvetlenül.
    url = database.AI_tools.OLLAMA_URL
    model_name = getattr(database.AI_tools, 'OLLAMA_MODEL_NAME', 'gpt-oss:20b') 
    payload = {"model": model_name, "prompt": prompt_text, "stream": False}
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        return response.json().get("response", "Nothing to analyze")
    else:
        return f"Error calling Ollama API: {response.status_code} - {response.text}"

In [3]:
mod_data = pd.read_csv(fr"{paths.ONEDRIVE}\test_wtd_for_summary.csv")

In [4]:
needed_formats = ['SuperMarkets', '5K+', '2-4K (CHM)', 'Express']

In [5]:
needed_departments = ['Home', 'Home Entertainment', 'Toys & Nursery', 'DIY', 'WIGIG',
       'Car', 'Gardening', 'Stationery', 'Electrical', 'Shopping bag',
       'Home seasonal']

In [6]:
mod_data = mod_data[mod_data['Department_name'].isin(needed_departments)]

In [7]:
mod_data = mod_data[mod_data['Store_format'].isin(needed_formats)]

In [8]:

mod_data['Range_Sales_excl_vat'] = mod_data['Sales_excl_vat'] - mod_data['Promo_sales_excl_vat'] - mod_data['Clearance_sales_excl_vat']
mod_data['Range_Sales_excl_vat_ly'] = mod_data['Sales_excl_vat_ly'] - mod_data['Promo_sales_excl_vat_ly'] - mod_data['Clearance_sales_excl_vat_ly']



# --- Segédfüggvény LFL számításhoz DataFrame oszlopokra ---
import numpy as np

def calculate_lfl_series(ty_sales_series, ly_sales_series):
    ty_sales_numeric = pd.to_numeric(ty_sales_series, errors='coerce').astype(float)
    ly_sales_numeric = pd.to_numeric(ly_sales_series, errors='coerce').astype(float)
    # 0 vagy NaN LY esetén None-t adunk vissza
    ly_sales_safe = ly_sales_numeric.replace(0, np.nan)
    result = ((ty_sales_numeric / ly_sales_safe) - 1) * 100
    return np.round(result, 2)

# --- Segédfüggvény LFL számításhoz skalár értékekre (összesített) ---
def calculate_lfl_scalar(ty_sales, ly_sales):
    if ly_sales is None or ly_sales == 0 or pd.isna(ly_sales):
        return pd.NA
    return round(((ty_sales / ly_sales) - 1)*100,2)

# --- 0. Teljes Vállalati Összesítés ---
if not mod_data.empty:
    overall_sales_ty = mod_data['Sales_excl_vat'].sum() # + mod_data['Consignment_sales_excl_vat'].sum()
    overall_sales_ly = mod_data['Sales_excl_vat_ly'].sum() # + mod_data['Consignment_sales_excl_vat_ly'].sum()
    overall_range_ty = mod_data['Range_Sales_excl_vat'].sum()
    overall_range_ly = mod_data['Range_Sales_excl_vat_ly'].sum()
    overall_promo_ty = mod_data['Promo_sales_excl_vat'].sum() 
    overall_promo_ly = mod_data['Promo_sales_excl_vat_ly'].sum()
    overall_clear_ty = mod_data['Clearance_sales_excl_vat'].sum()
    overall_clear_ly = mod_data['Clearance_sales_excl_vat_ly'].sum()

    overall_sales_lfl = calculate_lfl_scalar(overall_sales_ty, overall_sales_ly)
    overall_range_lfl = calculate_lfl_scalar(overall_range_ty, overall_range_ly)
    overall_promo_lfl = calculate_lfl_scalar(overall_promo_ty, overall_promo_ly)
    overall_clear_lfl = calculate_lfl_scalar(overall_clear_ty, overall_clear_ly)

    overall_summary_df = pd.DataFrame({
        'Metric': ['Overall Sales', 'Overall Range Sales', 'Overall Promotional Sales', 'Overall Clearance Sales'],
        'Sales_TY': [overall_sales_ty,overall_range_ty, overall_promo_ty, overall_clear_ty],
        'Sales_LY': [overall_sales_ly,overall_range_ly, overall_promo_ly, overall_clear_ly],
        'LFL': [overall_sales_lfl,overall_range_lfl, overall_promo_lfl, overall_clear_lfl]
    })
    overall_summary_df.fillna(0, inplace=True)
    overall_summary_str = overall_summary_df.to_json(orient='records')
else:
    overall_summary_str = "Teljes vállalati összesítés nem elérhető (üres bemeneti adat)."

# --- 1. Department Szintű Aggregáció ---
dept_agg = mod_data.groupby('Department_name').agg(
    Sales_TY=('Sales_excl_vat', 'sum'),
    Sales_LY=('Sales_excl_vat_ly', 'sum'),
    Range_TY =('Range_Sales_excl_vat', 'sum'),
    Range_LY=('Range_Sales_excl_vat_ly', 'sum'),
    Promo_Sales_TY=('Promo_sales_excl_vat', 'sum'),
    Promo_Sales_LY=('Promo_sales_excl_vat_ly', 'sum'),
    Clearance_Sales_TY=('Clearance_sales_excl_vat', 'sum'),
    Clearance_Sales_LY=('Clearance_sales_excl_vat_ly', 'sum')
).reset_index()

dept_agg['Sales_LFL'] = calculate_lfl_series(dept_agg['Sales_TY'], dept_agg['Sales_LY'])
dept_agg['Range_Sales_LFL'] = calculate_lfl_series(dept_agg['Range_TY'], dept_agg['Range_LY'])
dept_agg['Promo_Sales_LFL'] = calculate_lfl_series(dept_agg['Promo_Sales_TY'], dept_agg['Promo_Sales_LY'])
dept_agg['Clearance_Sales_LFL'] = calculate_lfl_series(dept_agg['Clearance_Sales_TY'], dept_agg['Clearance_Sales_LY'])
department_summary_df = dept_agg[['Department_name','Sales_LFL','Range_Sales_LFL', 'Promo_Sales_LFL', 'Clearance_Sales_LFL', 'Sales_TY', 'Sales_LY']]
department_summary_df.fillna(0, inplace=True)
department_summary_str = department_summary_df.to_json(orient='records')

# --- 2. Section Szintű Aggregáció ---
section_agg = mod_data.groupby(['Department_name', 'Section_name']).agg(
    Sales_TY=('Sales_excl_vat', 'sum'),
    Sales_LY=('Sales_excl_vat_ly', 'sum'),
    Range_TY=('Range_Sales_excl_vat', 'sum'),
    Range_LY=('Range_Sales_excl_vat_ly', 'sum'),
    Promo_Sales_TY=('Promo_sales_excl_vat', 'sum'),
    Promo_Sales_LY=('Promo_sales_excl_vat_ly', 'sum'),
    Clearance_Sales_TY=('Clearance_sales_excl_vat', 'sum'),
    Clearance_Sales_LY=('Clearance_sales_excl_vat_ly', 'sum')
).reset_index()

section_agg['Sales_LFL'] = calculate_lfl_series(section_agg['Sales_TY'], section_agg['Sales_LY'])
section_agg['Range_Sales_LFL'] = calculate_lfl_series(section_agg['Range_TY'], section_agg['Range_LY'])
section_agg['Promo_Sales_LFL'] = calculate_lfl_series(section_agg['Promo_Sales_TY'], section_agg['Promo_Sales_LY'])
section_agg['Clearance_Sales_LFL'] = calculate_lfl_series(section_agg['Clearance_Sales_TY'], section_agg['Clearance_Sales_LY'])
section_summary_df = section_agg[['Department_name', 'Section_name', 'Sales_LFL','Range_Sales_LFL', 'Promo_Sales_LFL', 'Clearance_Sales_LFL', 'Sales_TY', 'Sales_LY']]
section_summary_str = section_summary_df.fillna(0).to_json(orient='records')

# --- 3. Group Szintű Aggregáció ---
group_agg = mod_data.groupby(['Department_name', 'Section_name', 'Group_name']).agg(
    Sales_TY=('Sales_excl_vat', 'sum'),
    Sales_LY=('Sales_excl_vat_ly', 'sum'),
    Range_TY=('Range_Sales_excl_vat', 'sum'),
    Range_LY=('Range_Sales_excl_vat_ly', 'sum'),
    Promo_Sales_TY=('Promo_sales_excl_vat', 'sum'),
    Promo_Sales_LY=('Promo_sales_excl_vat_ly', 'sum'),
    Clearance_Sales_TY=('Clearance_sales_excl_vat', 'sum'),
    Clearance_Sales_LY=('Clearance_sales_excl_vat_ly', 'sum')
).reset_index()

group_agg['Sales_LFL'] = calculate_lfl_series(group_agg['Sales_TY'], group_agg['Sales_LY'])
group_agg['Range_Sales_LFL'] = calculate_lfl_series(group_agg['Range_TY'], group_agg['Range_LY'])
group_agg['Promo_Sales_LFL'] = calculate_lfl_series(group_agg['Promo_Sales_TY'], group_agg['Promo_Sales_LY'])
group_agg['Clearance_Sales_LFL'] = calculate_lfl_series(group_agg['Clearance_Sales_TY'], group_agg['Clearance_Sales_LY'])
group_summary_df = group_agg[['Department_name', 'Section_name', 'Group_name', 'Sales_LFL','Range_Sales_LFL', 'Promo_Sales_LFL', 'Clearance_Sales_LFL', 'Sales_TY', 'Sales_LY']]
group_summary_str = group_summary_df.fillna(0).to_json(orient='records')

# --- 4. Store Format Szintű Aggregáció ---
if 'Store_format' in mod_data.columns and not mod_data['Store_format'].empty:
    store_format_agg = mod_data.groupby('Store_format').agg(
        Sales_TY=('Sales_excl_vat', 'sum'),
        Sales_LY=('Sales_excl_vat_ly', 'sum'),
        Range_TY=('Range_Sales_excl_vat', 'sum'),
        Range_LY=('Range_Sales_excl_vat_ly', 'sum'),
        Promo_Sales_TY=('Promo_sales_excl_vat', 'sum'),
        Promo_Sales_LY=('Promo_sales_excl_vat_ly', 'sum'),
        Clearance_Sales_TY=('Clearance_sales_excl_vat', 'sum'),
        Clearance_Sales_LY=('Clearance_sales_excl_vat_ly', 'sum')
    ).reset_index()

    store_format_agg['Sales_LFL'] = calculate_lfl_series(store_format_agg['Sales_TY'], store_format_agg['Sales_LY'])
    store_format_agg['Range_Sales_LFL'] = calculate_lfl_series(store_format_agg['Range_TY'], store_format_agg['Range_LY'])
    store_format_agg['Promo_Sales_LFL'] = calculate_lfl_series(store_format_agg['Promo_Sales_TY'], store_format_agg['Promo_Sales_LY'])
    store_format_agg['Clearance_Sales_LFL'] = calculate_lfl_series(store_format_agg['Clearance_Sales_TY'], store_format_agg['Clearance_Sales_LY'])
    store_format_summary_df = store_format_agg[['Store_format', 'Sales_LFL','Range_Sales_LFL', 'Promo_Sales_LFL', 'Clearance_Sales_LFL', 'Sales_TY', 'Sales_LY']]
    store_format_summary_str = store_format_summary_df.fillna(0).to_json(orient='records')
else:
    store_format_summary_str = "Store format adat nem elérhető vagy a 'store_format' oszlop hiányzik/üres."

C:\Users\pmajor1\AppData\Local\Temp\ipykernel_20440\537749217.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  department_summary_df.fillna(0, inplace=True)


In [9]:
overall_summary_df

,Metric,Sales_TY,Sales_LY,LFL
0,Overall Sales,1.932708e+06,2.142579e+06,-9.80
1,Overall Range Sales,8.171031e+05,9.030353e+05,-9.52
2,Overall Promotional Sales,9.938328e+05,1.114111e+06,-10.80
3,Overall Clearance Sales,1.217718e+05,1.254326e+05,-2.92


In [20]:
base_prompt= """You are an expert financial analyst AI. Your task is to analyze the provided pre-aggregated financial summaries and generate a structured narrative report **in English, strictly following the specified analysis points and output format.**

**Data Context:**
*   The financial figures are in GBP.
*   Numerical values (including LFLs) are floats, using a dot (.) as the decimal separator. LFLs are presented as decimals and rounded to 2 decimals (e.g., 5.2 for +5,2%, -2.7 for -2,7%). NaN indicates missing or inapplicable LFL (e.g., due to zero Last Year sales).
*   The product hierarchy is: department -> section -> group.
*   Each summary contains pre-calculated Like-for-Like (LFL) figures for Sales, Promotional Sales, and Clearance Sales. They also include `Sales_TY` (This Year Sales) and `Sales_LY` (Last Year Sales) for context and weighting the importance of LFL figures. **Your analysis MUST primarily focus on these LFL figures.**

**Key Columns in the Summaries (example for Department Summary, similar for others):**
*   `Department_name`: Product department (or `section`, `group`, `store_format` in respective summaries)
*   `Sales_LFL`: Pre-calculated Sales Like-for-Like growth.
*   `Range_Sales_LFL`: Pre-calculated Range Sales Like-for-Like growth (Sales excluding promotions and clearance).
*   `Promo_Sales_LFL`: Pre-calculated Promotional Sales Like-for-Like growth.
*   `Clearance_Sales_LFL`: Pre-calculated Clearance Sales Like-for-Like growth.
*   `Sales_TY`: Total Sales This Year for the aggregated level.
*   `Sales_LY`: Total Sales Last Year for the aggregated level.
"""

In [26]:
format_prompt = """lease generate a concise yet comprehensive report based on these instructions. Focus on actionable insights and clearly reference which summary is used for each part of the analysis. Present LFL values as percentages with one decimal place (e.g., +3.5%, -2.1%, N/A). For every analysis point, provide a brief but informative textual evaluation, do not just list the numbers!

Please return your output as a clean, consistently structured English narrative.
Use the following formatting rules strictly:

Use numbered main sections (1., 2.) exactly as defined in the prompt.

Within each section, use bullet points (-) for subpoints.

Ensure the same structure is used for each Department_name.

"""

In [32]:
overview_prompt=f"""
**Analysis and Reporting Structure (provide the output in list format and structure it to bullet points, in English. Adhere strictly to this structure and ensure all points are addressed):**

1. Overall GM Performance (Based on 'Overall GM Summary:'):  
   - Report the overall GM `Sales_LFL`, `Range_Sales_LFL`, `Promo_Sales_LFL`, and `Clearance_Sales_LFL` (as percentages, one decimal place, e.g., 5.2 -> +5.2%, NaN -> "N/A").
   - Briefly comment on these figures, highlighting key trends and their potential business implications.

**Raw Input Data (as JSON):**
- `overall_gm_summary`:
{overall_summary_str}
Your colleague made the depamartment level analysis as well, so please build on that and do not repeat the same information.
- `department_summary`:{dep_result}"""

In [28]:
department_prompt =f""" Department Level Analysis (Based on 'Department Summary'):
   - List all `Department_names` that have a negative `Sales_LFL`.
   - For each `Department_name` with a negative `Sales_LFL`:
      - Analyze its `Range_Sales_LFL`,`Promo_Sales_LFL` and `Clearance_Sales_LFL` from the 'Department Summary'.
      - **Clearly determine and justify** whether the negative `Sales_LFL` is primarily driven by poor promotional performance (`Promo_Sales_LFL`), poor clearance performance (`Clearance_Sales_LFL`), poor range performance (`Range_Sales_LFL`) or a decline in base (non-promotional, non-clearance) sales. (Infer base sales LFL: if `Sales_LFL` is negative, but `Promo_Sales_LFL` and `Clearance_Sales_LFL` are not, or are less negative, then the issue lies with `Range_Sales_LFL`).
      - Consider the `Sales_TY` magnitude when weighting the significance of LFL shortfalls (e.g., a small negative LFL for a department with large `Sales_TY` can be significant).
      department_summary:
{department_summary_str}"""


In [11]:
print(overall_prompt)

You are an expert financial analyst AI. Your task is to analyze the provided pre-aggregated financial summaries and generate a structured narrative report **in English, strictly following the specified analysis points and output format.**

**Data Context:**
*   The financial figures are in GBP.
*   Numerical values (including LFLs) are floats, using a dot (.) as the decimal separator. LFLs are presented as decimals and rounded to 2 decimals (e.g., 5.2 for +5,2%, -2.7 for -2,7%). NaN indicates missing or inapplicable LFL (e.g., due to zero Last Year sales).
*   The product hierarchy is: department -> section -> group.
*   Each summary contains pre-calculated Like-for-Like (LFL) figures for Sales, Promotional Sales, and Clearance Sales. They also include `Sales_TY` (This Year Sales) and `Sales_LY` (Last Year Sales) for context and weighting the importance of LFL figures. **Your analysis MUST primarily focus on these LFL figures.**

**Key Columns in the Summaries (example for Department S

In [29]:
dep_result = utils.analyse_financial_summaries(base_prompt + "\n" + department_prompt + "\n" + format_prompt  )

In [30]:
print(dep_result)

**1. Department Level Analysis (Based on ‘Department Summary’)**  

- **Car**  
  - Sales LFL: **‑30.4 %** – a sharp overall decline.  
  - Range LFL: **‑37.6 %** – the most negative indicator; the core catalogue is the primary driver of the fall.  
  - Promo LFL: **‑11.4 %** – moderate dip in promotional volume.  
  - Clearance LFL: **‑33.6 %** – significant erosion in clearance sales.  
  - **Conclusion:** The base (range) sales are the principal issue; promotions and clearance are also weak but less influential.  
  - **Actionable Insight:** Re‑evaluate the core product mix and pricing strategy; consider targeted promotions to offset the range slump.

- **DIY**  
  - Sales LFL: **‑7.3 %** – modest decline relative to its £152k turnover.  
  - Range LFL: **‑1.7 %** – nearly flat, indicating the catalogue is largely stable.  
  - Promo LFL: **‑38.0 %** – severe drop in promotional sales.  
  - Clearance LFL: **+284.5 %** – exceptional clearance lift.  
  - **Conclusion:** The steep ne

In [33]:
overview_result = utils.analyse_financial_summaries(base_prompt + "\n" + overview_prompt + "\n" + format_prompt  )

In [34]:
print(overview_result)

**1. Overall GM Performance**  
- **Sales LFL:** –9.8 % – a sizeable contraction that signals a broad decline in revenue across the entire portfolio.  
- **Range Sales LFL:** –9.5 % – the core catalogue is the biggest contributor to the dip, indicating pricing or assortment issues.  
- **Promo Sales LFL:** –10.8 % – promotional activity is underperforming, further amplifying the sales decline.  
- **Clearance Sales LFL:** –2.9 % – clearance is the least affected segment, suggesting the discount channel remains relatively resilient.  
- **Interpretation:** The GM view shows that both the main range and promotional sales are dragging the business down; clearance is comparatively stable but cannot offset the broader slide. Strategic focus should be on revitalising the core range and boosting targeted promotional spend.

**2. Department‑Level Analysis**  
*(Based on the department summary, each bullet contains the key LFL figures followed by a concise evaluation and an actionable recommend

In [35]:
from core import utils, database, paths
import pandas as pd
import numpy as np

# AI endpoint és modell
AI_URL = database.AI_tools.OLLAMA_URL
MODEL_NAME = getattr(database.AI_tools, 'OLLAMA_MODEL_NAME', 'gemma:2b')

# --- 1. Adat beolvasása és szűrés ---
mod_data = pd.read_csv(fr"{paths.ONEDRIVE}\test_wtd_for_summary.csv")

needed_formats = ['SuperMarkets', '5K+', '2-4K (CHM)', 'Express']
needed_departments = [
    'Home', 'Home Entertainment', 'Toys & Nursery', 'DIY', 'WIGIG',
    'Car', 'Gardening', 'Stationery', 'Electrical', 'Shopping bag',
    'Home seasonal'
]

mod_data = mod_data[
    mod_data['Department_name'].isin(needed_departments) &
    mod_data['Store_format'].isin(needed_formats)
]

# --- 2. Range Sales számítása ---
mod_data['Range_Sales_excl_vat'] = (
    mod_data['Sales_excl_vat'] - 
    mod_data['Promo_sales_excl_vat'] - 
    mod_data['Clearance_sales_excl_vat']
)

mod_data['Range_Sales_excl_vat_ly'] = (
    mod_data['Sales_excl_vat_ly'] - 
    mod_data['Promo_sales_excl_vat_ly'] - 
    mod_data['Clearance_sales_excl_vat_ly']
)

# --- Segédfüggvények ---
def calculate_lfl_series(ty_sales_series, ly_sales_series):
    ty_sales_numeric = pd.to_numeric(ty_sales_series, errors='coerce').astype(float)
    ly_sales_numeric = pd.to_numeric(ly_sales_series, errors='coerce').astype(float)
    ly_sales_safe = ly_sales_numeric.replace(0, np.nan)
    result = ((ty_sales_numeric / ly_sales_safe) - 1) * 100
    return np.round(result, 2)

def calculate_lfl_scalar(ty_sales, ly_sales):
    if ly_sales is None or ly_sales == 0 or pd.isna(ly_sales):
        return pd.NA
    return round(((ty_sales / ly_sales) - 1) * 100, 2)

# --- 3. Összesített vállalati szint ---
if not mod_data.empty:
    overall_sales_ty = mod_data['Sales_excl_vat'].sum()
    overall_sales_ly = mod_data['Sales_excl_vat_ly'].sum()
    overall_range_ty = mod_data['Range_Sales_excl_vat'].sum()
    overall_range_ly = mod_data['Range_Sales_excl_vat_ly'].sum()
    overall_promo_ty = mod_data['Promo_sales_excl_vat'].sum()
    overall_promo_ly = mod_data['Promo_sales_excl_vat_ly'].sum()
    overall_clear_ty = mod_data['Clearance_sales_excl_vat'].sum()
    overall_clear_ly = mod_data['Clearance_sales_excl_vat_ly'].sum()

    overall_summary_df = pd.DataFrame({
        'Metric': ['Overall Sales', 'Overall Range Sales', 'Overall Promo Sales', 'Overall Clearance Sales'],
        'Sales_TY': [overall_sales_ty, overall_range_ty, overall_promo_ty, overall_clear_ty],
        'Sales_LY': [overall_sales_ly, overall_range_ly, overall_promo_ly, overall_clear_ly],
        'Sales_LFL': [
            calculate_lfl_scalar(overall_sales_ty, overall_sales_ly),
            calculate_lfl_scalar(overall_range_ty, overall_range_ly),
            calculate_lfl_scalar(overall_promo_ty, overall_promo_ly),
            calculate_lfl_scalar(overall_clear_ty, overall_clear_ly)
        ]
    }).fillna(0)
    overall_summary_str = overall_summary_df.to_json(orient='records')
else:
    overall_summary_str = "N/A"

# --- 4. Department szint ---
dept_agg = mod_data.groupby('Department_name').agg(
    Sales_TY=('Sales_excl_vat', 'sum'),
    Sales_LY=('Sales_excl_vat_ly', 'sum'),
    Range_TY=('Range_Sales_excl_vat', 'sum'),
    Range_LY=('Range_Sales_excl_vat_ly', 'sum'),
    Promo_TY=('Promo_sales_excl_vat', 'sum'),
    Promo_LY=('Promo_sales_excl_vat_ly', 'sum'),
    Clear_TY=('Clearance_sales_excl_vat', 'sum'),
    Clear_LY=('Clearance_sales_excl_vat_ly', 'sum')
).reset_index()

dept_agg['Sales_LFL'] = calculate_lfl_series(dept_agg['Sales_TY'], dept_agg['Sales_LY'])
dept_agg['Range_LFL'] = calculate_lfl_series(dept_agg['Range_TY'], dept_agg['Range_LY'])
dept_agg['Promo_LFL'] = calculate_lfl_series(dept_agg['Promo_TY'], dept_agg['Promo_LY'])
dept_agg['Clear_LFL'] = calculate_lfl_series(dept_agg['Clear_TY'], dept_agg['Clear_LY'])

# 🔹 Csak negatív Sales_LFL és sorbarendezés
dept_neg_sorted = dept_agg[dept_agg['Sales_LFL'] < 0].sort_values('Sales_TY', ascending=False)

department_summary_str = dept_neg_sorted[
    ['Department_name', 'Sales_LFL', 'Range_LFL', 'Promo_LFL', 'Clear_LFL', 'Sales_TY']
].to_json(orient='records')

# --- 5. Extra statisztika ---
neg_total_ty = dept_neg_sorted['Sales_TY'].sum()
neg_avg_lfl = dept_neg_sorted['Sales_LFL'].mean() if not dept_neg_sorted.empty else 0
neg_count = dept_neg_sorted.shape[0]
neg_ratio = round((neg_total_ty / overall_sales_ty) * 100, 1) if overall_sales_ty else 0

neg_stats_str = f"""
Negative LFL Departments Stats:
- Total TY Sales (negative LFL departments): £{neg_total_ty:,.0f}
- Average Sales_LFL among them: {neg_avg_lfl:.1f}%
- Count of negative LFL departments: {neg_count}
- Share of total GM Sales: {neg_ratio}%
"""

# --- 6. Prompt szabálylogikával ---
final_prompt = f"""
You are an expert retail financial analyst AI.

Analyze the provided aggregated GM sales summaries and produce a concise, structured English report with two sections:

1. Overall GM Performance
   - List Sales_LFL, Range_Sales_LFL, Promo_Sales_LFL, Clearance_Sales_LFL (% one decimal, +x.x%, NaN -> "N/A").
   - Comment on overall trends and potential business implications.

2. Department Level Negative LFL Analysis
   - Departments are pre-filtered to those with negative Sales_LFL, sorted by Sales_TY (highest first).
   - For each, determine the **main driver of decline** and explain the likely cause based strictly on the following decision rules:

Decision rules:
- Range Driver: if Range_LFL is negative and more negative than both Promo_LFL and Clear_LFL.
- Promo Driver: if Promo_LFL is negative and significantly more negative than Range_LFL or Clear_LFL.
- Clearance Driver: if Clear_LFL is negative and significantly more negative than Range_LFL and Promo_LFL.
- Combined Factors: if two or more LFL values are significantly negative (similar magnitude).

Additional instructions:
- Only identify and explain the likely cause — DO NOT provide solutions or action recommendations.
- Consider the magnitude of Sales_TY in weighing significance.

**Extra Context:**
{neg_stats_str}

**Data:**
overall_summary: {overall_summary_str}
department_summary_sorted_negatives: {department_summary_str}

**Format rules:**
- Use numbered sections as above.
- Bullet points inside each section.
- One decimal for percentages.
- Compact but clear commentary.
- Cause analysis based solely on provided decision rules.
"""

# --- 7. AI hívás ---
result = utils.analyse_financial_summaries(final_prompt)
print(result)

**1. Overall GM Performance**  
- **Sales LFL:** ‑9.8 %  
- **Range Sales LFL:** ‑9.5 %  
- **Promo Sales LFL:** ‑10.8 %  
- **Clearance Sales LFL:** ‑3.0 %  

*Commentary*  
- All core sales segments are down year‑over‑year, with the **promo channel experiencing the steepest decline**.  
- The **range channel** is also contracting, suggesting weaker core merchandise performance.  
- **Clearance sales** are the least affected, indicating that inventory liquidation is still relatively effective.  
- The overall downward trend signals a need to reassess pricing, assortment, and promotional mix, but the magnitude of the decline varies across categories.

---

**2. Department Level Negative LFL Analysis**  
*(Departments sorted by FY sales – highest first)*  

| Department | Main Driver of Decline | Likely Cause (based on rules) |
|------------|-----------------------|--------------------------------|
| **Toys & Nursery** | Promo | Promo LFL (‑7.3 %) is the most negative value; range and c

In [37]:
/how to export the final_prompt string to a file?

# --- 8. Export to file ---
with open('report.txt', 'w') as f:
    f.write(result)

print("Report exported to report.txt")

Report exported to report.txt


Docstring: Alias for `%%writefile`.
File:      c:\users\pmajor1\appdata\local\anaconda3\envs\my_env\lib\site-packages\ipython\core\magic.py